#### Inisialiate Spark

In [7]:
import sys, glob
spark_lib = '/opt/spark/python/lib'
for z in glob.glob(spark_lib + '/*.zip'):
    sys.path.insert(0, z)
sys.path.insert(0, '/opt/spark/python')

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, abs as spark_abs, round as spark_round, lit, median

spark = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .appName("SilverCleaningNB") \
    .getOrCreate()

df_train = spark.read.csv("hdfs://namenode:8020/data/bronze/home_credit/raw/application_train.csv", header=True, inferSchema=True)
print("Train count:", df_train.count())
df_train.show(5, truncate=False)

Train count: 307511
+----------+------+------------------+-----------+------------+---------------+------------+----------------+----------+-----------+---------------+---------------+----------------+-----------------------------+--------------------+-----------------+--------------------------+----------+-------------+-----------------+---------------+-----------+----------+--------------+---------------+----------------+----------+----------+---------------+---------------+--------------------+---------------------------+--------------------------+-----------------------+--------------------------+--------------------------+---------------------------+----------------------+----------------------+-----------------------+----------------------+-------------------+------------------+-------------------+--------------+----------------+---------------------------+------------------+--------------+-------------+-------------+-------------+-------------+------------+--------------------+-

#### Imputasi

In [8]:
def fill_missing_with_median(df, col_name):
    med = df.select(median(col(col_name))).collect()[0][0]
    if med is not None:
        df = df.withColumn(col_name, when(col(col_name).isNull(), lit(med)).otherwise(col(col_name)))
    return df

def fill_missing_with_mode(df, col_name, default="Unknown"):
    mode_row = df.groupBy(col_name).count().orderBy("count", ascending=False).first()
    mode_val = mode_row[col_name] if mode_row and mode_row[col_name] is not None else default
    df = df.withColumn(col_name, when(col(col_name).isNull(), lit(mode_val)).otherwise(col(col_name)))
    return df

#### Transformasi

In [9]:

df_train = df_train.withColumn("AGE_YEARS", spark_round(spark_abs(col("DAYS_BIRTH")) / 365.25, 2))
df_train = df_train.drop("DAYS_BIRTH")

df_train = df_train.withColumn("FLAG_UNEMPLOYED", when(col("DAYS_EMPLOYED") == 365243, 1).otherwise(0))
df_train = df_train.withColumn("YEARS_EMPLOYED", 
                   when(col("DAYS_EMPLOYED") == 365243, 0.0)
                   .otherwise(spark_round(spark_abs(col("DAYS_EMPLOYED")) / 365.25, 2)))
df_train = df_train.drop("DAYS_EMPLOYED")

for flag_col in ["FLAG_OWN_CAR", "FLAG_OWN_REALTY"]:
    if flag_col in df_train.columns:
        df_train = df_train.withColumn(flag_col, when(col(flag_col) == "Y", 1).otherwise(0).cast("int"))

# Lihat hasil sementara
df_train.select("AGE_YEARS", "YEARS_EMPLOYED", "FLAG_UNEMPLOYED", "FLAG_OWN_CAR", "FLAG_OWN_REALTY").show(5)

+---------+--------------+---------------+------------+---------------+
|AGE_YEARS|YEARS_EMPLOYED|FLAG_UNEMPLOYED|FLAG_OWN_CAR|FLAG_OWN_REALTY|
+---------+--------------+---------------+------------+---------------+
|     25.9|          1.74|              0|           0|              1|
|     45.9|          3.25|              0|           0|              0|
|    52.15|          0.62|              0|           1|              1|
|    52.03|          8.32|              0|           0|              1|
|    54.57|          8.32|              0|           0|              1|
+---------+--------------+---------------+------------+---------------+
only showing top 5 rows



#### Imputasi Missing Value

In [11]:
from pyspark.sql.functions import col, when, count, lit
from pyspark.sql.types import DoubleType

def fill_missing_with_median_fast(df, col_name):
    # Hitung perkiraan median (lebih cepat)
    q = df.select(col_name).approxQuantile(col_name, [0.5], 0.01)
    med = q[0] if q else None
    if med is not None:
        df = df.withColumn(col_name, when(col(col_name).isNull(), lit(med)).otherwise(col(col_name)))
    return df

def fill_missing_with_mode(df, col_name, default="Unknown"):
    mode_row = df.groupBy(col_name).count().orderBy("count", ascending=False).first()
    mode_val = mode_row[col_name] if mode_row and mode_row[col_name] is not None else default
    df = df.withColumn(col_name, when(col(col_name).isNull(), lit(mode_val)).otherwise(col(col_name)))
    return df

numeric_cols = [c for c, t in df_train.dtypes if t in ["double", "int", "float"] and c not in ("TARGET", "SK_ID_CURR")]
for nc in numeric_cols:
    if df_train.filter(col(nc).isNull()).count() > 0:
        df_train = fill_missing_with_median_fast(df_train, nc)

# Kategorikal
cat_cols = [c for c, t in df_train.dtypes if t == "string"]
for cc in cat_cols:
    if df_train.filter(col(cc).isNull()).count() > 0:
        df_train = fill_missing_with_mode(df_train, cc)

# Verifikasi
nulls_after = df_train.select([count(when(col(c).isNull(), c)).alias(c) for c in numeric_cols[:5]])
nulls_after.show()

+------------+---------------+------------+----------------+----------+
|FLAG_OWN_CAR|FLAG_OWN_REALTY|CNT_CHILDREN|AMT_INCOME_TOTAL|AMT_CREDIT|
+------------+---------------+------------+----------------+----------+
|           0|              0|           0|               0|         0|
+------------+---------------+------------+----------------+----------+



#### Dedpulikasi & Save Stagging

In [12]:
count_before = df_train.count()
df_train = df_train.dropDuplicates(["SK_ID_CURR"])
count_after = df_train.count()
print(f"Sebelum dedup: {count_before}, sesudah: {count_after}")

# Simpan ke staging
df_train.write.mode("overwrite").parquet("hdfs://namenode:8020/data/silver/staging/application_train_clean")
print("Train clean saved.")

Sebelum dedup: 307511, sesudah: 307511


Train clean saved.


#### Aplication_test.csv

In [13]:
# Baca test
df_test = spark.read.csv("hdfs://namenode:8020/data/bronze/home_credit/raw/application_test.csv", header=True, inferSchema=True)
print("Test count:", df_test.count())

# Terapkan semua langkah yang sama untuk df_test
# 1. Konversi DAYS
df_test = df_test.withColumn("AGE_YEARS", spark_round(spark_abs(col("DAYS_BIRTH")) / 365.25, 2))
df_test = df_test.drop("DAYS_BIRTH")
df_test = df_test.withColumn("FLAG_UNEMPLOYED", when(col("DAYS_EMPLOYED") == 365243, 1).otherwise(0))
df_test = df_test.withColumn("YEARS_EMPLOYED", 
                 when(col("DAYS_EMPLOYED") == 365243, 0.0)
                 .otherwise(spark_round(spark_abs(col("DAYS_EMPLOYED")) / 365.25, 2)))
df_test = df_test.drop("DAYS_EMPLOYED")

# 2. Flag konversi
for flag_col in ["FLAG_OWN_CAR", "FLAG_OWN_REALTY"]:
    if flag_col in df_test.columns:
        df_test = df_test.withColumn(flag_col, when(col(flag_col) == "Y", 1).otherwise(0).cast("int"))

# 3. Imputasi
test_num_cols = [c for c, t in df_test.dtypes if t in ["double", "int", "float"] and c not in ("TARGET", "SK_ID_CURR")]
for nc in test_num_cols:
    if df_test.filter(col(nc).isNull()).count() > 0:
        df_test = fill_missing_with_median(df_test, nc)

test_cat_cols = [c for c, t in df_test.dtypes if t == "string"]
for cc in test_cat_cols:
    if df_test.filter(col(cc).isNull()).count() > 0:
        df_test = fill_missing_with_mode(df_test, cc)

# 4. Dedup
df_test = df_test.dropDuplicates(["SK_ID_CURR"])

# 5. Simpan
df_test.write.mode("overwrite").parquet("hdfs://namenode:8020/data/silver/staging/application_test_clean")
print("Test clean saved.")

Test count: 48744


26/08/10 09:11:02 ERROR TransportClient: Failed to send RPC RPC 6691010577660721789 to /172.18.0.9:43120: io.netty.channel.StacklessClosedChannelException
io.netty.channel.StacklessClosedChannelException
	at io.netty.channel.AbstractChannel$AbstractUnsafe.write(Object, ChannelPromise)(Unknown Source)
26/08/10 09:11:02 ERROR TaskSchedulerImpl: Lost executor 2 on 172.18.0.9: Command exited with code 137
26/08/10 09:11:02 WARN BlockManagerMasterEndpoint: Error trying to remove broadcast 1066 from block manager BlockManagerId(2, 172.18.0.9, 42141, None)
java.io.IOException: Failed to send RPC RPC 6691010577660721789 to /172.18.0.9:43120: io.netty.channel.StacklessClosedChannelException
	at org.apache.spark.network.client.TransportClient$RpcChannelListener.handleFailure(TransportClient.java:395)
	at org.apache.spark.network.client.TransportClient$StdChannelListener.operationComplete(TransportClient.java:372)
	at io.netty.util.concurrent.DefaultPromise.notifyListener0(DefaultPromise.java:590

Test clean saved.


#### Verif

In [14]:
df_train_check = spark.read.parquet("hdfs://namenode:8020/data/silver/staging/application_train_clean")
print("Train clean count:", df_train_check.count())
df_train_check.select("SK_ID_CURR", "TARGET", "AGE_YEARS", "YEARS_EMPLOYED", "FLAG_UNEMPLOYED").show(5)

# Cek test clean
df_test_check = spark.read.parquet("hdfs://namenode:8020/data/silver/staging/application_test_clean")
print("Test clean count:", df_test_check.count())
df_test_check.select("SK_ID_CURR", "AGE_YEARS", "YEARS_EMPLOYED", "FLAG_UNEMPLOYED").show(5)

Train clean count: 307511
+----------+------+---------+--------------+---------------+
|SK_ID_CURR|TARGET|AGE_YEARS|YEARS_EMPLOYED|FLAG_UNEMPLOYED|
+----------+------+---------+--------------+---------------+
|    100002|     1|     25.9|          1.74|              0|
|    100029|     0|     30.9|          2.04|              0|
|    100046|     0|    44.15|          4.82|              0|
|    100052|     0|    21.83|          2.45|              0|
|    100079|     0|    42.05|          0.31|              0|
+----------+------+---------+--------------+---------------+
only showing top 5 rows

Test clean count: 48744
+----------+---------+--------------+---------------+
|SK_ID_CURR|AGE_YEARS|YEARS_EMPLOYED|FLAG_UNEMPLOYED|
+----------+---------+--------------+---------------+
|    100057|    45.68|          7.06|              0|
|    100067|    28.46|          7.19|              0|
|    100091|    33.62|          2.54|              0|
|    100092|     53.9|           9.8|              0